In [0]:

CREATE OR REPLACE VIEW
pondolens.dev_gold.vw_deped_school_age_spending
AS

WITH school_age_population AS (
    SELECT
        fp.year_id,
        fp.location_id,
        SUM(fp.population_count) AS school_age_population
    FROM pondolens.dev_gold.fact_population fp
    INNER JOIN pondolens.dev_gold.dim_age_group ag
        ON fp.age_group_id = ag.age_group_id
    WHERE ag.age_group_id IN (80001, 80002)
      AND fp.population_count IS NOT NULL
    GROUP BY
        fp.year_id,
        fp.location_id
    HAVING COUNT(DISTINCT fp.age_group_id) = 2
),

deped_obligations AS (
    SELECT
        fe.year_id,
        fe.location_id,
        SUM(fe.obligation_amount) AS deped_obligations
    FROM pondolens.dev_gold.fact_regional_expenditure fe
    INNER JOIN pondolens.dev_gold.dim_particular dp
        ON fe.particular_id = dp.particular_id
    WHERE UPPER(TRIM(dp.category)) = 'DEPARTMENTS'
      AND UPPER(TRIM(dp.particular))
          LIKE 'DEPARTMENT OF EDUCATION%'
    GROUP BY
        fe.year_id,
        fe.location_id
)

SELECT
    dy.year,
    dl.location_id,
    dl.location_code,
    dl.location,
    p.school_age_population,
    d.deped_obligations,

    ROUND(
        d.deped_obligations
        / NULLIF(p.school_age_population, 0),
        2
    ) AS deped_obligations_per_child

FROM school_age_population p
INNER JOIN deped_obligations d
    ON p.year_id = d.year_id
   AND p.location_id = d.location_id
INNER JOIN pondolens.dev_gold.dim_location dl
    ON p.location_id = dl.location_id
INNER JOIN pondolens.dev_gold.dim_year dy
    ON p.year_id = dy.year_id

WHERE p.school_age_population > 0
  AND UPPER(TRIM(dl.location_code)) <> 'PHL'
  AND dy.year BETWEEN 2021 AND 2025;
